# Week 3 Mini Project: Iris Flower Clustering Project

**Topic:** Unsupervised Learning - K-Means Clustering and PCA  
**Objective:** Perform K-Means clustering on the Iris dataset, visualize clusters, apply PCA to reduce dimensions, and compare predicted clusters with true species labels.

Dataset: `uciml/iris` from Kaggle via `kagglehub`.

> Run the cells from top to bottom. The first run may require internet access and Kaggle authentication, depending on your environment.


## 1. Install required libraries
If needed, run the following in a notebook cell. In Google Colab/Jupyter, remove the `#` at the start to install packages.


In [ ]:
# %pip install kagglehub[pandas-datasets] pandas scikit-learn matplotlib seaborn


## 2. Load the Iris dataset
The Kaggle dataset usually contains `Iris.csv`. If loading fails, check the dataset files and update `file_path`.


In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from kagglehub import KaggleDatasetAdapter
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

file_path = "Iris.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "uciml/iris",
    file_path,
)

print("First five records:")
display(df.head())
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())


## 3. Inspect and prepare the data
The Iris features are numeric measurements. The true label column is kept only for comparison after clustering, not for training K-Means.


In [ ]:
df = df.copy()
df.columns = [column.strip() for column in df.columns]

label_column = "Species" if "Species" in df.columns else df.columns[-1]
id_columns = [column for column in df.columns if column.lower() in ["id", "index"]]

feature_columns = [
    column for column in df.select_dtypes(include=["number"]).columns
    if column not in id_columns
]

X = df[feature_columns].copy()
true_labels = df[label_column].astype(str)

print("Label column:", label_column)
print("Feature columns used for clustering:", feature_columns)
print("Missing values in selected features:")
display(X.isna().sum())
print("Species counts:")
display(true_labels.value_counts())


## 4. Scale features
K-Means uses distance, so feature scaling helps each measurement contribute fairly.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scaled_df = pd.DataFrame(X_scaled, columns=feature_columns)
display(scaled_df.head())


## 5. Apply K-Means clustering with k = 3
The assignment asks for three clusters because the Iris dataset has three known species.


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
predicted_clusters = kmeans.fit_predict(X_scaled)

df["Predicted_Cluster"] = predicted_clusters

print("Cluster counts:")
display(df["Predicted_Cluster"].value_counts().sort_index())
print(f"K-Means inertia: {kmeans.inertia_:.2f}")
print(f"Silhouette score: {silhouette_score(X_scaled, predicted_clusters):.4f}")


## 6. Apply PCA to reduce dimensions
PCA reduces the four Iris measurement columns into two principal components for visualization.


In [ ]:
pca = PCA(n_components=2, random_state=42)
principal_components = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(principal_components, columns=["PC1", "PC2"])
pca_df["Predicted_Cluster"] = predicted_clusters
pca_df["True_Label"] = true_labels.values

print("Explained variance ratio by PCA components:")
print(f"PC1: {pca.explained_variance_ratio_[0]:.4f}")
print(f"PC2: {pca.explained_variance_ratio_[1]:.4f}")
print(f"Total explained variance: {pca.explained_variance_ratio_.sum():.4f}")
display(pca_df.head())


## 7. Visualize predicted clusters
This scatter plot shows the K-Means clusters after reducing the data to two PCA components.


In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Predicted_Cluster",
    palette="Set2",
    s=80,
    edgecolor="black",
)
plt.title("K-Means Clusters on Iris Dataset (PCA View)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()


## 8. Compare predicted clusters with true labels
K-Means is unsupervised, so cluster numbers do not automatically match species names. The table and side-by-side plots compare the discovered clusters with the actual Iris species.


In [ ]:
comparison_table = pd.crosstab(
    pca_df["True_Label"],
    pca_df["Predicted_Cluster"],
    rownames=["True Species"],
    colnames=["Predicted Cluster"],
)

display(comparison_table)

label_encoder = LabelEncoder()
encoded_true_labels = label_encoder.fit_transform(true_labels)

print(f"Adjusted Rand Index: {adjusted_rand_score(encoded_true_labels, predicted_clusters):.4f}")
print(f"Normalized Mutual Information: {normalized_mutual_info_score(encoded_true_labels, predicted_clusters):.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Predicted_Cluster",
    palette="Set2",
    s=75,
    edgecolor="black",
    ax=axes[0],
)
axes[0].set_title("Predicted K-Means Clusters")
axes[0].legend(title="Cluster")

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="True_Label",
    palette="Set1",
    s=75,
    edgecolor="black",
    ax=axes[1],
)
axes[1].set_title("True Iris Species")
axes[1].legend(title="Species")

for axis in axes:
    axis.set_xlabel("Principal Component 1")
    axis.set_ylabel("Principal Component 2")

plt.tight_layout()
plt.show()


## 9. Sample results
The final table shows a few rows with their original species and predicted cluster.


In [ ]:
final_results = df[feature_columns + [label_column, "Predicted_Cluster"]].copy()
display(final_results.head(15))


## Conclusion
K-Means clustering with `k=3` was applied to the Iris dataset after scaling the numeric flower measurements. PCA reduced the feature space to two dimensions, making it possible to visualize the clusters in a scatter plot. The predicted clusters were compared with the true Iris species using a crosstab, Adjusted Rand Index, and Normalized Mutual Information.

**Submission note:** Run every notebook cell and save the notebook with the output tables and graphs visible before submitting.
